# Cleaning The Data
This notebook covers how to first clean and filter the data, then how to discritize it.

### 1. Filtering and cleaning
First lets clean the data by:
- Selecting only the desired columns
- Filtering what values we want in those columns to decrease the number of variables
- Remove rows with empty variables

In [210]:
import pandas as pd
from sklearn.model_selection import train_test_split

data_path = "../data/survey_results_public.csv"
df = pd.read_csv(data_path)

# Only keep the wanted columns listed below:
cols = [
    "Country",
    "OrgSize",
    "DevType",
    "AISelect",
    "YearsCode",
    "YearsCodePro",
    "CodingActivities",
    "ConvertedCompYearly"
]
df = df[cols].copy()
# Print dimensions
print(f"Shape: {df.shape}")

Shape: (89184, 8)


In [211]:
# Filter data to ensure each field only has a certain value
# Country: United States of America
# Org: 500 - 999 employees
# Role: Developer

filtered = df[
    (df["Country"] == "United States of America") &
    (df["OrgSize"] == ("1,000 to 4,999 employees")) &
    (df["DevType"].str.contains("Developer"))
].copy()

# Show shape post filtering
print(f"Shape post filtering: {filtered.shape}")

Shape post filtering: (1285, 8)


In [212]:
# Remove rows with missing values
filtered = filtered.dropna().copy()

# Show shpae after removing
print(f"Shape post drop missing values: {filtered.shape}")

Shape post drop missing values: (1106, 8)


### 2. Discritization
The next step is to descritize the data. This will include:
- Converting strings to numerical values
- Discretize coding years into: low, medium, high
- Discretize salary into: low, medium, high, very_high
- Simplify AI usage and remove outliers
- Simplify coding activites
- Simplify developer type

In [ ]:
# Convert strings into numerical 

# Helper function to convert string to number
def convert_years(x):
    if x == "More than 50 years":
        return 51
    elif x == "Less than 1 year":
        return 0.5
    else:
        return float(x)
filtered["YearsCode_num"] = filtered["YearsCode"].apply(convert_years)
filtered["YearsCodePro_num"] = filtered["YearsCodePro"].apply(convert_years)

,YearsCode,YearsCode_num,YearsCodePro,YearsCodePro_num
6,4,4.0,3,3.0
42,21,21.0,16,16.0
273,11,11.0,7,7.0
536,16,16.0,14,14.0
959,30,30.0,28,28.0


In [214]:
# Discretize number of years coding into three categories

# Helper function to discretize
def discretize_years(years):
    if years < 3:
        return "low"
    elif years < 10:
        return "medium"
    else:
        return "high"
    
# Descritize both the pro and the non pro coding years
filtered["YearsCode_cat"] = filtered["YearsCode_num"].apply(discretize_years)
filtered["YearsCodePro_cat"] = filtered["YearsCodePro_num"].apply(discretize_years)

# Show counts in each category
print(filtered["YearsCode_cat"].value_counts())
print()
print(filtered["YearsCodePro_cat"].value_counts())

YearsCode_cat
high      830
medium    274
low         2
Name: count, dtype: int64

YearsCodePro_cat
high      596
medium    422
low        88
Name: count, dtype: int64


In [215]:
# Simplify salary into 4 buckets: low, medium, high, very high based on quartiles
def discretize_salary(data):
    data = data.copy()
    
    q1 = data['ConvertedCompYearly'].quantile(0.25)
    q2 = data['ConvertedCompYearly'].quantile(0.5)
    q3 = data['ConvertedCompYearly'].quantile(0.75)

    salary_labels = []
    
    for val in data['ConvertedCompYearly']:
        if val < q1:
            salary_labels.append("low")
        elif val >= q1 and val < q2:
            salary_labels.append("medium")
        elif val >= q2 and val < q3:
            salary_labels.append("high")
        else:
            salary_labels.append("very_high")

    data = data.copy()
    data["ConvertedCompYearly_cat"] = salary_labels
    return data

filtered = discretize_salary(filtered)
filtered.head()

,Country,OrgSize,DevType,AISelect,YearsCode,YearsCodePro,CodingActivities,ConvertedCompYearly,YearsCode_num,YearsCodePro_num,YearsCode_cat,YearsCodePro_cat,ConvertedCompYearly_cat
6,United States of America,"1,000 to 4,999 employees","Developer, full-stack",Yes,4,3,Hobby;Contribute to open-source projects;Profe...,135000.0,4.0,3.0,medium,medium,medium
42,United States of America,"1,000 to 4,999 employees","Developer, front-end",Yes,21,16,Hobby;Freelance/contract work,165000.0,21.0,16.0,high,high,high
273,United States of America,"1,000 to 4,999 employees","Developer, full-stack","No, and I don't plan to",11,7,I don’t code outside of work,240000.0,11.0,7.0,high,medium,very_high
536,United States of America,"1,000 to 4,999 employees","Developer, mobile","No, and I don't plan to",16,14,Hobby;Contribute to open-source projects,140000.0,16.0,14.0,high,high,medium
959,United States of America,"1,000 to 4,999 employees","Developer, back-end","No, and I don't plan to",30,28,Hobby;Professional development or self-paced l...,160000.0,30.0,28.0,high,high,high


In [216]:
# Simplify the devloper titles into easier to see categories
def simplify_type_of_dev(dev):
    dev = str(dev)
    
    if "Developer, full-stack" in dev:
        return "full_stack"
    elif "Developer, back-end" in dev:
        return "back_end"
    elif "Developer, front-end" in dev:
        return "front_end"
    elif "Developer, mobile" in dev:
        return "mobile"
    elif "Developer, desktop or enterprise applications" in dev:
        return "applications"
    elif "Developer, embedded applications or devices" in dev:
        return "enbedded"
    elif "Developer, QA or test" in dev:
        return "qa"
    elif "Experience" in dev:
        return "experience"
    elif "Developer, game or graphics" in dev:
        return "game_graphics"
    elif "Developer Advocate" in dev:
        return "advocate"
    else:
        return "other_developer"
filtered["DevType_cat"] = filtered["DevType"].apply(simplify_type_of_dev)

print(filtered["DevType_cat"].value_counts())

DevType_cat
full_stack       512
back_end         304
applications      80
front_end         77
enbedded          44
mobile            38
qa                19
experience        14
game_graphics     11
advocate           7
Name: count, dtype: int64


In [217]:
# Simplify list of hobbies into hobby vs no hobby
def simplify_hobbies(hobby):
    hobby = str(hobby)
    if "Hobby" in hobby:
        return "codes_as_hobby"
    else:
        return "no_code_as_hobby"
filtered["CodingActivities_cat"] = filtered["CodingActivities"].apply(simplify_hobbies)
print(filtered["CodingActivities_cat"].value_counts())

CodingActivities_cat
codes_as_hobby      784
no_code_as_hobby    322
Name: count, dtype: int64


In [218]:
# Simplify the AI field for easier use 
def simplify_AI(ai):
    ai = str(ai)
    if "No" and "but" in ai:
        return "plan_to"
    elif "No" in ai:
        return "no"
    else:
        return "yes"
filtered["AISelect_cat"] = filtered["AISelect"].apply(simplify_AI)
print(filtered["AISelect_cat"].value_counts())

AISelect_cat
no         422
yes        388
plan_to    296
Name: count, dtype: int64


In [219]:
# Simplify the org field for readability 
def simplify_org(ai):
    return "medium_size"
filtered["OrgSize_cat"] = filtered["OrgSize"].apply(simplify_org)
print(filtered["OrgSize_cat"].value_counts())

OrgSize_cat
medium_size    1106
Name: count, dtype: int64


In [220]:
# Simplify the country field for readability 
def simplify_country(ai):
    return "USA"
filtered["Country_cat"] = filtered["Country"].apply(simplify_country)
print(filtered["Country_cat"].value_counts())

Country_cat
USA    1106
Name: count, dtype: int64


### 3. Create Final Bayesian Network Dataset

In [221]:
# Remove the extra columns created when discretizing the data and set the correct
# names to the columns
bn_data = pd.DataFrame({
    "Country": filtered["Country_cat"],
    "OrgSize": filtered["OrgSize_cat"],
    "DevType": filtered["DevType_cat"],
    "AISelect": filtered["AISelect_cat"],
    "YearsCode": filtered["YearsCode_cat"],
    "YearsCodePro": filtered["YearsCodePro_cat"],
    "CodingActivities": filtered["CodingActivities_cat"],
    "ConvertedCompYearly": filtered["ConvertedCompYearly_cat"],
})
print(f"Final Shape: {bn_data.shape}")
bn_data.head()

Final Shape: (1106, 8)


,Country,OrgSize,DevType,AISelect,YearsCode,YearsCodePro,CodingActivities,ConvertedCompYearly
6,USA,medium_size,full_stack,yes,medium,medium,codes_as_hobby,medium
42,USA,medium_size,front_end,yes,high,high,codes_as_hobby,high
273,USA,medium_size,full_stack,no,high,medium,no_code_as_hobby,very_high
536,USA,medium_size,mobile,no,high,high,codes_as_hobby,medium
959,USA,medium_size,back_end,no,high,high,codes_as_hobby,high


In [222]:
# Save the final data into a csv
bn_data.to_csv("../data/cleaned_stackoverflow_bn_data.csv", index=False)